In [0]:
%sql
ALTER TABLE adbdevbankproject.silver.mcc_codes_new
RENAME TO adbdevbankproject.silver.mcc_codes;

In [0]:
%sql
DROP TABLE adbdevbankproject.silver.mcc_codes;

In [0]:
%sql
DESCRIBE DETAIL adbdevbankproject.silver.mcc_codes_new;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,94cc1e05-57d1-4af1-b715-fa95624f6b28,adbdevbankproject.silver.mcc_codes_new,null,abfss://silver@stgdevbankproject.dfs.core.windows.net/mcc_codes_new,2026-08-21T14:06:39.328Z,2026-08-21T14:06:39.000Z,List(),List(),1,3868,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT *
FROM adbdevbankproject.silver.mcc_codes_new
LIMIT 10;

mcc_code,merchant_category
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


In [0]:
%sql
CREATE TABLE adbdevbankproject.silver.mcc_codes_new
USING DELTA
LOCATION 'abfss://silver@stgdevbankproject.dfs.core.windows.net/mcc_codes_new'
AS
SELECT *
FROM adbdevbankproject.silver.mcc_codes;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE DETAIL adbdevbankproject.silver.mcc_codes;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,54795af1-128c-4858-a995-b27b18e01b07,adbdevbankproject.silver.mcc_codes,null,,2026-08-21T14:02:42.981Z,2026-08-21T14:02:45.000Z,List(),List(),1,2913,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT *
FROM adbdevbankproject.silver.mcc_codes
LIMIT 20;

mcc_code,merchant_category
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


In [0]:
# Save MCC mapping as a Silver Delta table

df_mcc_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("adbdevbankproject.silver.mcc_codes")

In [0]:
# Transform MCC mapping from wide format to long format

from pyspark.sql.functions import col, lit, explode, array, struct

# Read the Bronze MCC table
df_mcc_raw = spark.table("adbdevbankproject.bronze.mcc_codes")

# Get all MCC code columns
mcc_columns = df_mcc_raw.columns

# Convert all MCC columns into rows
df_mcc_silver = df_mcc_raw.select(
    explode(
        array(*[
            struct(
                lit(int(c)).alias("mcc_code"),
                col(c).alias("merchant_category")
            )
            for c in mcc_columns
        ])
    ).alias("mcc")
).select(
    "mcc.mcc_code",
    "mcc.merchant_category"
)

display(df_mcc_silver)

mcc_code,merchant_category
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products
3007,Coated and Laminated Products
3008,Steel Drums and Barrels
3009,Fabricated Structural Metal Products
3058,"Tools, Parts, Supplies Manufacturing"
3066,Miscellaneous Metals


In [0]:
# Convert MCC JSON mapping into a structured DataFrame

df_mcc_raw = spark.table("adbdevbankproject.bronze.mcc_codes")

display(df_mcc_raw)

1711,3000,3001,3005,3006,3007,3008,3009,3058,3066,3075,3132,3144,3174,3256,3260,3359,3387,3389,3390,3393,3395,3405,3504,3509,3596,3640,3684,3722,3730,3771,3775,3780,4111,4112,4121,4131,4214,4411,4511,4722,4784,4814,4829,4899,4900,5045,5094,5192,5193,5211,5251,5261,5300,5310,5311,5411,5499,5533,5541,5621,5651,5655,5661,5712,5719,5722,5732,5733,5812,5813,5814,5815,5816,5912,5921,5932,5941,5942,5947,5970,5977,6300,7011,7210,7230,7276,7349,7393,7531,7538,7542,7549,7801,7802,7832,7922,7995,7996,8011,8021,8041,8043,8049,8062,8099,8111,8931,9402
"Heating, Plumbing, Air Conditioning Contractors",Steelworks,Steel Products Manufacturing,Miscellaneous Metal Fabrication,Miscellaneous Fabricated Metal Products,Coated and Laminated Products,Steel Drums and Barrels,Fabricated Structural Metal Products,"Tools, Parts, Supplies Manufacturing",Miscellaneous Metals,"Bolt, Nut, Screw, Rivet Manufacturing",Leather Goods,Floor Covering Stores,Upholstery and Drapery Stores,"Brick, Stone, and Related Materials",Pottery and Ceramics,Non-Ferrous Metal Foundries,"Electroplating, Plating, Polishing Services",Non-Precious Metal Services,Miscellaneous Metalwork,Heat Treating Metal Services,Welding Repair,Ironwork,Gardening Supplies,Industrial Equipment and Supplies,Miscellaneous Machinery and Parts Manufacturing,"Lighting, Fixtures, Electrical Supplies",Semiconductors and Related Devices,Passenger Railways,Ship Chandlers,Railroad Passenger Transport,Railroad Freight,Computer Network Services,Local and Suburban Commuter Transportation,Passenger Railways,Taxicabs and Limousines,Bus Lines,Motor Freight Carriers and Trucking,Cruise Lines,Airlines,Travel Agencies,Tolls and Bridge Fees,Telecommunication Services,Money Transfer,"Cable, Satellite, and Other Pay Television Services","Utilities - Electric, Gas, Water, Sanitary","Computers, Computer Peripheral Equipment",Precious Stones and Metals,"Books, Periodicals, Newspapers","Florists Supplies, Nursery Stock and Flowers",Lumber and Building Materials,Hardware Stores,Lawn and Garden Supply Stores,Wholesale Clubs,Discount Stores,Department Stores,"Grocery Stores, Supermarkets",Miscellaneous Food Stores,Automotive Parts and Accessories Stores,Service Stations,Women's Ready-To-Wear Stores,Family Clothing Stores,"Sports Apparel, Riding Apparel Stores",Shoe Stores,"Furniture, Home Furnishings, and Equipment Stores",Miscellaneous Home Furnishing Stores,Household Appliance Stores,Electronics Stores,Music Stores - Musical Instruments,Eating Places and Restaurants,Drinking Places (Alcoholic Beverages),Fast Food Restaurants,"Digital Goods - Media, Books, Apps",Digital Goods - Games,Drug Stores and Pharmacies,"Package Stores, Beer, Wine, Liquor",Antique Shops,Sporting Goods Stores,Book Stores,"Gift, Card, Novelty Stores","Artist Supply Stores, Craft Shops",Cosmetic Stores,"Insurance Sales, Underwriting","Lodging - Hotels, Motels, Resorts",Laundry Services,Beauty and Barber Shops,Tax Preparation Services,Cleaning and Maintenance Services,"Detective Agencies, Security Services",Automotive Body Repair Shops,Automotive Service Shops,Car Washes,Towing Services,"Athletic Fields, Commercial Sports","Recreational Sports, Clubs",Motion Picture Theaters,Theatrical Producers,"Betting (including Lottery Tickets, Casinos)","Amusement Parks, Carnivals, Circuses","Doctors, Physicians",Dentists and Orthodontists,Chiropractors,"Optometrists, Optical Goods and Eyeglasses",Podiatrists,Hospitals,Medical Services,Legal Services and Attorneys,"Accounting, Auditing, and Bookkeeping Services",Postal Services - Government Only
